# Temperature Prediction in London Using ML Models

**Goal:** Predict the daily mean temperature in London using historical weather data.

**Pipeline:**
1. Load & clean historical weather data
2. Exploratory Data Analysis
3. Feature engineering (time features + lag features)
4. Train and compare multiple regression models via scikit-learn Pipelines
5. Track experiments with MLflow
6. Evaluate best model on hold-out test set

In [ ]:
import sys
sys.path.insert(0, '../src')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
import mlflow.sklearn

from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from data_preprocessing import load_data, engineer_features, clean_data, get_X_y
from models import build_pipelines

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

## 1. Load & Preprocess Data

In [ ]:
df_raw = load_data('../data/london_weather.csv')
print(f'Shape: {df_raw.shape}')
df_raw.head()

In [ ]:
df_raw.info()
df_raw.describe()

In [ ]:
missing = df_raw.isnull().sum()
missing[missing > 0].sort_values(ascending=False)

## 2. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# Temperature over time
axes[0, 0].plot(df_raw['date'], df_raw['mean_temp'], linewidth=0.6, alpha=0.8)
axes[0, 0].set_title('Daily Mean Temperature Over Time')
axes[0, 0].set_ylabel('Temperature (°C)')

# Monthly distribution
df_raw['month'] = df_raw['date'].dt.month
monthly = df_raw.groupby('month')['mean_temp'].mean()
axes[0, 1].bar(monthly.index, monthly.values, color=sns.color_palette('coolwarm', 12))
axes[0, 1].set_title('Average Temperature by Month')
axes[0, 1].set_xlabel('Month')
axes[0, 1].set_ylabel('Mean Temp (°C)')

# Distribution of target
axes[1, 0].hist(df_raw['mean_temp'].dropna(), bins=40, color='steelblue', edgecolor='white')
axes[1, 0].set_title('Distribution of Mean Temperature')
axes[1, 0].set_xlabel('Temperature (°C)')

# Correlation heatmap
numeric = df_raw.select_dtypes(include=np.number).drop(columns=['month'])
corr = numeric.corr()[['mean_temp']].drop('mean_temp').sort_values('mean_temp', ascending=False)
sns.heatmap(corr, ax=axes[1, 1], annot=True, fmt='.2f', cmap='RdBu_r', center=0, linewidths=0.5)
axes[1, 1].set_title('Feature Correlation with Mean Temp')

plt.tight_layout()
plt.savefig('../outputs/eda_overview.png', bbox_inches='tight')
plt.show()

## 3. Feature Engineering

In [ ]:
df = engineer_features(df_raw.copy())
df = clean_data(df)
print(f'After feature engineering: {df.shape}')
df[['date', 'mean_temp', 'month_sin', 'month_cos', 'mean_temp_lag1', 'mean_temp_roll7']].head(10)

In [ ]:
X, y = get_X_y(df)
print(f'Features: {X.shape[1]}  |  Samples: {X.shape[0]}')
print('Feature columns:', X.columns.tolist())

## 4. Train / Test Split (time-series aware)

In [ ]:
split = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]
print(f'Train size: {len(X_train)}  |  Test size: {len(X_test)}')

## 5. Model Training & MLflow Tracking

In [ ]:
mlflow.set_experiment('london_temperature_prediction')
tscv = TimeSeriesSplit(n_splits=5)
pipelines = build_pipelines()
results = []

for name, pipeline in pipelines.items():
    with mlflow.start_run(run_name=name):
        cv_rmse = -cross_val_score(
            pipeline, X_train, y_train,
            cv=tscv, scoring='neg_root_mean_squared_error', n_jobs=-1
        )
        pipeline.fit(X_train, y_train)
        y_pred = pipeline.predict(X_test)

        metrics = {
            'model': name,
            'cv_rmse': round(cv_rmse.mean(), 4),
            'test_mae': round(mean_absolute_error(y_test, y_pred), 4),
            'test_rmse': round(np.sqrt(mean_squared_error(y_test, y_pred)), 4),
            'test_r2': round(r2_score(y_test, y_pred), 4),
        }
        results.append(metrics)

        mlflow.log_params({'model': name})
        mlflow.log_metrics({k: v for k, v in metrics.items() if k != 'model'})
        mlflow.sklearn.log_model(pipeline, artifact_path='model')

results_df = pd.DataFrame(results).sort_values('test_rmse')
results_df

## 6. Model Comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
metrics_to_plot = ['test_rmse', 'test_mae', 'test_r2']
titles = ['RMSE (lower is better)', 'MAE (lower is better)', 'R² (higher is better)']
colors = sns.color_palette('Set2', len(results_df))

for ax, metric, title in zip(axes, metrics_to_plot, titles):
    bars = ax.barh(results_df['model'], results_df[metric], color=colors)
    ax.set_title(title)
    ax.bar_label(bars, fmt='%.3f', padding=3)
    ax.set_xlim(0, results_df[metric].max() * 1.2)

plt.tight_layout()
plt.savefig('../outputs/model_comparison.png', bbox_inches='tight')
plt.show()

## 7. Best Model — Predictions vs Actuals

In [ ]:
best_name = results_df.iloc[0]['model']
best_pipeline = pipelines[best_name]
y_pred_best = best_pipeline.predict(X_test)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Predicted vs actual scatter
axes[0].scatter(y_test, y_pred_best, alpha=0.4, s=10, color='steelblue')
lims = [min(y_test.min(), y_pred_best.min()), max(y_test.max(), y_pred_best.max())]
axes[0].plot(lims, lims, 'r--', linewidth=1.5)
axes[0].set_title(f'{best_name}: Predicted vs Actual')
axes[0].set_xlabel('Actual (°C)')
axes[0].set_ylabel('Predicted (°C)')

# Time-series of predictions over last 365 days of test set
n = min(365, len(y_test))
axes[1].plot(range(n), y_test.values[-n:], label='Actual', linewidth=1.2)
axes[1].plot(range(n), y_pred_best[-n:], label='Predicted', linewidth=1.2, linestyle='--')
axes[1].set_title('Predictions vs Actuals (last 365 test days)')
axes[1].set_xlabel('Day')
axes[1].set_ylabel('Temperature (°C)')
axes[1].legend()

plt.tight_layout()
plt.savefig('../outputs/best_model_predictions.png', bbox_inches='tight')
plt.show()

print(f'Best model: {best_name}')
print(results_df[results_df['model'] == best_name].to_string(index=False))

## 8. Feature Importance (for tree-based best model)

In [ ]:
model_step = best_pipeline.named_steps['model']
if hasattr(model_step, 'feature_importances_'):
    importances = pd.Series(model_step.feature_importances_, index=X.columns)
    importances.sort_values().plot(kind='barh', figsize=(8, 6), color='teal')
    plt.title(f'Feature Importances — {best_name}')
    plt.xlabel('Importance')
    plt.tight_layout()
    plt.savefig('../outputs/feature_importance.png', bbox_inches='tight')
    plt.show()
else:
    print(f'{best_name} does not expose feature importances directly.')